# Notebook 02 — Operator KB & Paper Ingestion

**Goal**:
1. Load the FASTEXPR operator knowledge base from YAML and index it into Chroma
2. Ingest research papers / notes into the `papers` Chroma collection
3. Demonstrate RAG retrieval across all knowledge sources

**Requires**: Notebook 01 completed (vector store initialized)

In [1]:
import time
import traceback

t0 = time.perf_counter()
last = t0

print("[1/6] importing nest_asyncio ...")
import nest_asyncio
now = time.perf_counter()
print(f"      done in {now - last:.2f}s")
last = now

print("[2/6] applying nest_asyncio ...")
nest_asyncio.apply()
now = time.perf_counter()
print(f"      done in {now - last:.2f}s")
last = now

print("[3/6] importing settings ...")
from alpha_agent.config import settings
now = time.perf_counter()
print(f"      done in {now - last:.2f}s")
last = now

print("[4/6] importing VectorStore ...")
from alpha_agent.knowledge.vector_store import VectorStore
now = time.perf_counter()
print(f"      done in {now - last:.2f}s")
last = now

print("[5/6] building VectorStore (this may download/load embedding model on first run) ...")
print(f"      embed_model = {settings.embed_model}")
print(f"      chroma_dir  = {settings.chroma_persist_dir}")

try:
    store = VectorStore()
    now = time.perf_counter()
    print(f"      done in {now - last:.2f}s")
    last = now

    print("[6/6] sanity check: count operators collection ...")
    from alpha_agent.knowledge.vector_store import COLLECTION_OPERATORS
    op_count = store.count(COLLECTION_OPERATORS)
    now = time.perf_counter()
    print(f"      done in {now - last:.2f}s (operators={op_count})")

    total = now - t0
    print(f"\nVector store loaded. total elapsed = {total:.2f}s")
except Exception:
    now = time.perf_counter()
    print(f"\nFAILED after {now - t0:.2f}s")
    traceback.print_exc()
    raise

[1/6] importing nest_asyncio ...
      done in 0.00s
[2/6] applying nest_asyncio ...
      done in 0.00s
[3/6] importing settings ...
      done in 0.26s
[4/6] importing VectorStore ...
      done in 1.05s
[5/6] building VectorStore (this may download/load embedding model on first run) ...
      embed_model = all-MiniLM-L6-v2
      chroma_dir  = /Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/chroma


/Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/myenv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


      done in 13.42s
[6/6] sanity check: count operators collection ...
      done in 0.09s (operators=36)

Vector store loaded. total elapsed = 14.82s


## 1. Load and index FASTEXPR operators

In [2]:
from alpha_agent.data.operator_kb import OperatorKB

kb = OperatorKB()
print(f"Loaded {len(kb.operators)} operators from YAML")

# Index into Chroma
n_indexed = kb.index_into_store(store, force_refresh=False)
print(f"Indexed {n_indexed} operators (0 = already up to date)")

# Show categories
from collections import Counter
cats = Counter(op['category'] for op in kb.operators)
for cat, count in sorted(cats.items()):
    print(f"  {cat}: {count}")

Loaded 60 operators from YAML
Indexed 60 operators (0 = already up to date)
  conditional: 5
  cross_section: 6
  group: 6
  math: 15
  time_series: 24
  transform: 2
  vector: 2


In [3]:
# Search operators by research concept
from alpha_agent.knowledge.vector_store import COLLECTION_OPERATORS

query = "rolling correlation between two signals"
results = store.query(COLLECTION_OPERATORS, query, k=4)

print(f"Operator search: '{query}'\n")
for r in results:
    print(f"  [{r['metadata']['name']}] dist={r['distance']:.3f}")
    print(f"  {r['document'][:200]}\n")

Operator search: 'rolling correlation between two signals'

  [correlation] dist=0.517
  Operator: correlation
Category: time_series
Signature: correlation(x, y, d)
Returns: MATRIX
Description: Alias for ts_corr. Rolling correlation between x and y.
Example: correlation(close, volume, 10)

  [ts_corr] dist=0.587
  Operator: ts_corr
Category: time_series
Signature: ts_corr(x, y, d)
Returns: MATRIX
Description: Rolling Pearson correlation.
Example: ts_corr(returns, volume, 20)
Tips: Very sensitive to short window

  [ts_covariance] dist=0.648
  Operator: ts_covariance
Category: time_series
Signature: ts_covariance(y, x, d)
Returns: MATRIX
Description: Rolling covariance between y and x.
Example: ts_covariance(returns, volume, 20)
Tips: Scale

  [ts_av_diff] dist=0.666
  Operator: ts_av_diff
Category: time_series
Signature: ts_av_diff(x, d)
Returns: MATRIX
Description: Difference from rolling mean (NaN-aware).
Example: ts_av_diff(returns, 20)
Tips: Similar to x - ts_m



In [4]:
# Print compact prompt-ready operator reference
print(kb.to_prompt_text(categories=['time_series', 'cross_section']))

FASTEXPR Operator Reference:
  days_from_last_change(x) → MATRIX: Number of days since x last changed.
  hump(x, hump=0.01) → MATRIX: Limits daily jump magnitude in x.
  kth_element(x, d, k, ignore='NaN') → MATRIX: K-th value in lookback window.
  last_diff_value(x, d) → MATRIX: Most recent value differing from current within d days.
  ts_arg_max(x, d) → MATRIX: Days since max over past d days.
  ts_arg_min(x, d) → MATRIX: Days since min over past d days.
  ts_av_diff(x, d) → MATRIX: Difference from rolling mean (NaN-aware).
  ts_backfill(x, lookback=d, k=1) → MATRIX: Backfills NaN with recent valid observations.
  ts_corr(x, y, d) → MATRIX: Rolling Pearson correlation.
  ts_count_nans(x, d) → MATRIX: Counts NaNs in rolling window.
  ts_covariance(y, x, d) → MATRIX: Rolling covariance between y and x.
  ts_decay_linear(x, d, dense=false) → MATRIX: Linearly decayed average over d days.
  ts_delay(x, d) → MATRIX: Lagged value d days ago.
  ts_delta(x, d) → MATRIX: Change over d days.
  t

## 2. Ingest research papers / notes

Place PDF or Markdown files in `data/papers/` and run the cells below.
You can also ingest plain-text snippets directly.

In [5]:
from alpha_agent.knowledge.paper_ingest import PaperIngest

pi = PaperIngest(store)

# Example: ingest a known factor insight as raw text
example_text = """
Cross-sectional momentum in equities (Jegadeesh & Titman 1993):
Stocks that performed well over the past 6-12 months tend to continue performing well
over the next 3-6 months (momentum). Conversely, short-term (1-month) returns
tend to reverse (short-term reversal). Momentum is stronger for:
- High volume, high liquidity stocks
- Stocks with analyst coverage
- Mid-cap stocks (less crowding vs. large-cap)

Volatility-adjusted momentum (Daniel & Moskowitz 2016):
Scale momentum exposure by 1/variance. This dramatically improves risk-adjusted returns
by reducing exposure during momentum crashes (typically high volatility periods).
FASTEXPR implementation: rank(ts_mean(returns, 252)) * (1 / ts_std_dev(returns, 60))

Earnings momentum / PEAD (Bernard & Thomas 1989):
Post-earnings announcement drift — stocks with positive earnings surprises continue
to drift up for several months. Can be proxied by analyst revision momentum.
"""

n = pi.ingest_text(example_text, source="momentum_notes")
print(f"Ingested {n} chunks from example text")

Ingested 1 chunks from example text


In [6]:
# Optionally: ingest all PDFs from a directory
import pathlib

papers_dir = pathlib.Path("../data/papers")
if papers_dir.exists():
    results = pi.ingest_directory(papers_dir)
    for fname, count in results.items():
        print(f"  {fname}: {count} chunks")
else:
    print(f"No papers directory found at {papers_dir}. Create it and add PDFs/MDs.")

No papers directory found at ../data/papers. Create it and add PDFs/MDs.


## 3. Cross-collection RAG demo

In [7]:
# Query across all collections simultaneously
query = "volatility scaling for momentum"
multi_results = store.query_multi(query, k_per_col=3)

print(f"Multi-collection search: '{query}'\n")
for r in multi_results[:9]:
    coll = r.get('collection', '?')
    dist = r.get('distance', 0)
    doc = r.get('document', '')[:150]
    print(f"[{coll}] dist={dist:.3f}")
    print(f"  {doc}")
    print()

Multi-collection search: 'volatility scaling for momentum'

[papers] dist=0.630
  Cross-sectional momentum in equities (Jegadeesh & Titman 1993): Stocks that performed well over the past 6-12 months tend to continue performing well 

[papers] dist=0.683
  # Recommended Alpha Research Papers This note groups recommended papers for the WorldQuant Brain Alpha project. ## Price and Volume (pv1) 1. Jegadeesh

[operators] dist=0.713
  Operator: ts_scale
Category: time_series
Signature: ts_scale(x, d, constant=0)
Returns: MATRIX
Description: Scales x to [0, 1] within rolling window.


[operators] dist=0.741
  Operator: scale
Category: cross_section
Signature: scale(x, scale=1, longscale=1, shortscale=1)
Returns: MATRIX
Description: Scales book to target abs

[operators] dist=0.747
  Operator: group_scale
Category: group
Signature: group_scale(x, group)
Returns: MATRIX
Description: Scales values to [0, 1] within group.
Example: gro

[papers] dist=0.747
  Often there is no best answer for this 

In [8]:
# Optional: release DB/file handles before switching notebooks
import gc

for obj_name in ["memory", "registry", "store", "pi"]:
    obj = globals().get(obj_name)
    close_fn = getattr(obj, "close", None)
    if callable(close_fn):
        try:
            close_fn()
            print(f"Closed: {obj_name}")
        except Exception as e:
            print(f"Failed to close {obj_name}: {e}")

for obj_name in ["memory", "registry", "store", "pi"]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()
print("Released DB/file handles and cleared related globals.")

Released DB/file handles and cleared related globals.


## ✅ Notebook 02 Complete

You now have all knowledge indexed:
- Data fields (from NB01)
- FASTEXPR operators
- Research papers / notes

**Next**: Run `03_idea_to_expression_demo.ipynb` to see the full idea→expression pipeline without WQB.